# Montagem — Dataset Final Treino V4

Consolida a base V3 (balanced) com novas fontes próprias do CheckAI:
- **Pipeline Autoral Verdadeiro** — claims factuais rastreáveis com `label=1`
- **Google Fact Check expandido** — claims verificados com `label=0/1` (ROTULO_FORTE)

**Saídas:**
- `dataset_final_treino_v4_full`
- `dataset_final_treino_v4_balanced`
- `dataset_final_treino_v4_text_control`

> Esta etapa é apenas montagem. **Nenhum modelo é treinado aqui.**
> V1, V2 e V3 **não são alteradas**.

In [1]:
import re
import uuid
import pandas as pd
from datetime import datetime
from pathlib import Path

## Configuração

In [2]:
SEED      = 42
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

_cwd         = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'src' else _cwd

PASTA_V3      = PROJECT_ROOT / 'dados' / 'dataset_unificado' / 'final'
PASTA_AUTORAL = PROJECT_ROOT / 'dados' / 'pipeline_checkai_autoral' / 'curated'
PASTA_GFC     = PROJECT_ROOT / 'dados' / 'pipeline_falso_google_factcheck' / 'curated'
PASTA_SAIDA   = PROJECT_ROOT / 'dados' / 'dataset_unificado' / 'final'

print(f'TIMESTAMP    : {TIMESTAMP}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')

TIMESTAMP    : 2026-05-30_23-29-02
PROJECT_ROOT : C:\Users\offan\Desktop\ml-checkai


## Funções auxiliares

In [3]:
def _mais_recente(pasta: Path, padrao: str) -> Path:
    candidatos = sorted([p for p in pasta.glob(padrao) if p.stat().st_size > 0])
    if not candidatos:
        raise FileNotFoundError(f'Nenhum arquivo: {pasta}/{padrao}')
    return candidatos[-1]

def normalizar_texto(t) -> str:
    if not isinstance(t, str):
        return ''
    t = t.lower().strip()
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'[^\w\s]', '', t)
    return t

def calcular_faixa(n) -> str:
    try:
        n = int(float(n))
    except (TypeError, ValueError):
        return ''
    if n < 500:   return 'curto'
    if n < 2000:  return 'medio'
    if n < 5000:  return 'longo'
    return 'muito_longo'

COLUNAS_FINAIS = [
    'id_registro', 'texto_principal', 'texto_principal_modelo',
    'label', 'label_detalhe', 'pipeline_origem', 'dataset_origem',
    'portal_origem', 'origem_texto', 'origem_qualidade',
    'fonte_dataset', 'referencia_dataset', 'url_origem', 'data_publicacao',
    'tema', 'subtema', 'tamanho_chars', 'tamanho_chars_modelo',
    'faixa_tamanho_modelo', 'status_curadoria', 'motivo_status',
    'metodo_coleta', 'query_matched', 'tema_query', 'subtema_query',
]
print(f'Helpers ok. Schema final: {len(COLUNAS_FINAIS)} colunas')

Helpers ok. Schema final: 25 colunas


## Fonte 1 — V3 Balanced (base histórica)

Carrega o arquivo mais recente `dataset_final_treino_v3_balanced_*.csv`.
A V3 **não é alterada** — é apenas lida e incorporada na V4.

In [4]:
ARQ_V3 = _mais_recente(PASTA_V3, 'dataset_final_treino_v3_balanced_*.csv')
df_v3  = pd.read_csv(ARQ_V3, encoding='utf-8-sig', dtype=str, low_memory=False)

print(f'V3 balanced : {ARQ_V3.name}')
print(f'Shape       : {df_v3.shape}')
print(f'label dist  : {df_v3["label"].value_counts().to_dict()}')
print(f'dataset_orig: {df_v3["dataset_origem"].value_counts().head(5).to_dict()}')

V3 balanced : dataset_final_treino_v3_balanced_2026-05-26_23-54-08.csv
Shape       : (10340, 19)
label dist  : {'1': 5170, '0': 5170}
dataset_orig: {'FAKEBR_TEXT_NORMALIZED': 7018, 'FAKETRUEBR': 2832, 'V2_PROPRIA': 490}


## Fonte 2 — CheckAI Autoral Verdadeiro

Filtros obrigatórios:
- `status_curadoria = APROVADO_AUTO`
- `tipo_claim = FATO_INSTITUCIONAL`
- `label = 1`
- `texto_principal_modelo` e `url_origem` preenchidos

In [5]:
ARQ_AUT    = _mais_recente(PASTA_AUTORAL, 'checkai_autoral_verdadeiros_curated_*.csv')
df_aut_raw = pd.read_csv(ARQ_AUT, encoding='utf-8-sig', dtype=str, low_memory=False)

print(f'Autoral raw : {ARQ_AUT.name}')
print(f'Shape bruto : {df_aut_raw.shape}')
print(f'status_cur  : {df_aut_raw["status_curadoria"].value_counts().to_dict()}')
print(f'tipo_claim  : {df_aut_raw["tipo_claim"].value_counts().to_dict()}')

mask_aut = (
    (df_aut_raw['status_curadoria'] == 'APROVADO_AUTO')
    & (df_aut_raw['tipo_claim'] == 'FATO_INSTITUCIONAL')
    & (df_aut_raw['label'].fillna('') == '1')
    & (df_aut_raw['texto_principal_modelo'].fillna('').str.strip() != '')
    & (df_aut_raw['url_origem'].fillna('').str.strip() != '')
)
df_aut = df_aut_raw[mask_aut].copy().reset_index(drop=True)

print(f'\nApós filtro : {len(df_aut)} registros (de {len(df_aut_raw)})')
print(f'Descartados : {len(df_aut_raw) - len(df_aut)}')

Autoral raw : checkai_autoral_verdadeiros_curated_2026-05-30_21-04-39.csv
Shape bruto : (149, 28)
status_cur  : {'APROVADO_AUTO': 114, 'PENDENTE_REVISAO': 30, 'DESCARTADO': 5}
tipo_claim  : {'FATO_INSTITUCIONAL': 114, 'DECLARACAO_PUBLICA': 22, 'CHAMADA_EXPLICATIVA': 5, 'PAGINA_ESTATICA': 4, 'OUTRO_PENDENTE': 4}

Após filtro : 114 registros (de 149)
Descartados : 35


## Fonte 3 — Google Fact Check expandido

Filtros obrigatórios:
- `status_curadoria = APROVADO_AUTO`
- `origem_qualidade = ROTULO_FORTE`
- `label_detalhe` em {FALSO, ENGANOSO, FORA_DE_CONTEXTO, GFC_VERDADEIRO}
- `texto_principal` e `url_origem` preenchidos

> Sanitização defensiva: `url_consulta` removida se presente (pode conter API key).

In [6]:
_gfc_arqs = sorted([
    p for p in PASTA_GFC.glob('google_factcheck_curated_*.csv')
    if 'TESTE' not in p.name and p.stat().st_size > 0
])
ARQ_GFC    = _gfc_arqs[-1]
df_gfc_raw = pd.read_csv(ARQ_GFC, encoding='utf-8-sig', dtype=str, low_memory=False)

# Sanitização defensiva
if 'url_consulta' in df_gfc_raw.columns:
    df_gfc_raw = df_gfc_raw.drop(columns=['url_consulta'])
    print('Sanitização: url_consulta removida')

if 'status_curadoria' not in df_gfc_raw.columns:
    df_gfc_raw['status_curadoria'] = 'APROVADO_AUTO'
if 'origem_qualidade' not in df_gfc_raw.columns:
    df_gfc_raw['origem_qualidade'] = 'ROTULO_FORTE'

print(f'GFC curated : {ARQ_GFC.name}')
print(f'Shape bruto : {df_gfc_raw.shape}')

_LABELS_GFC = {'FALSO', 'ENGANOSO', 'FORA_DE_CONTEXTO', 'GFC_VERDADEIRO'}
mask_gfc = (
    (df_gfc_raw['status_curadoria'] == 'APROVADO_AUTO')
    & (df_gfc_raw['origem_qualidade'] == 'ROTULO_FORTE')
    & (df_gfc_raw['label_detalhe'].fillna('').isin(_LABELS_GFC))
    & (df_gfc_raw['texto_principal'].fillna('').str.strip() != '')
    & (df_gfc_raw['url_origem'].fillna('').str.strip() != '')
)
df_gfc = df_gfc_raw[mask_gfc].copy().reset_index(drop=True)

print(f'Após filtro : {len(df_gfc)} (de {len(df_gfc_raw)})')
print(f'label_detalhe: {df_gfc["label_detalhe"].value_counts().to_dict()}')
print(f'label        : {df_gfc["label"].value_counts().to_dict()}')

GFC curated : google_factcheck_curated_2026-05-30_21-49-09.csv
Shape bruto : (5208, 22)
Após filtro : 5145 (de 5208)
label_detalhe: {'FALSO': 3664, 'ENGANOSO': 1301, 'FORA_DE_CONTEXTO': 145, 'GFC_VERDADEIRO': 35}
label        : {'0.0': 5110, '1.0': 35}


## Harmonização de Schema

Adapta cada bloco ao schema final de 25 colunas.
Colunas ausentes são preenchidas com valores padrão coerentes.
Nenhum dado original é alterado.

In [7]:
# ─── V3 ────────────────────────────────────────────────────────────────
df_h_v3 = df_v3.copy()
for col, val in [
    ('tema', ''), ('subtema', ''),
    ('status_curadoria', 'APROVADO_AUTO'), ('motivo_status', 'herdado_v3'),
    ('query_matched', ''), ('tema_query', ''), ('subtema_query', ''),
]:
    if col not in df_h_v3.columns:
        df_h_v3[col] = val
if 'metodo_coleta' not in df_h_v3.columns:
    df_h_v3['metodo_coleta'] = df_h_v3.get('pipeline_origem', 'datasets_academicos')
df_h_v3['tamanho_chars_modelo'] = df_h_v3['texto_principal_modelo'].fillna('').str.len().astype(str)
df_h_v3['faixa_tamanho_modelo'] = df_h_v3['tamanho_chars_modelo'].apply(calcular_faixa)
df_h_v3['label'] = pd.to_numeric(df_h_v3['label'], errors='coerce').fillna(-1).astype(int)
for col in COLUNAS_FINAIS:
    if col not in df_h_v3.columns:
        df_h_v3[col] = ''
df_h_v3 = df_h_v3[COLUNAS_FINAIS].copy()
df_h_v3['_fonte_bloco'] = 'V3'

# ─── Autoral ────────────────────────────────────────────────────────────
df_h_aut = df_aut.copy()
df_h_aut['tamanho_chars'] = df_h_aut['texto_principal'].fillna('').str.len().astype(str)
df_h_aut['tamanho_chars_modelo'] = df_h_aut['texto_principal_modelo'].fillna('').str.len().astype(str)
df_h_aut['faixa_tamanho_modelo'] = df_h_aut['tamanho_chars_modelo'].apply(calcular_faixa)
df_h_aut['label'] = pd.to_numeric(df_h_aut['label'], errors='coerce').fillna(-1).astype(int)
for col in COLUNAS_FINAIS:
    if col not in df_h_aut.columns:
        df_h_aut[col] = ''
df_h_aut = df_h_aut[COLUNAS_FINAIS].copy()
df_h_aut['_fonte_bloco'] = 'AUTORAL'

# ─── GFC ────────────────────────────────────────────────────────────────
df_h_gfc = df_gfc.copy()
df_h_gfc['texto_principal_modelo'] = df_h_gfc['texto_principal'].fillna('').str[:1500].str.strip()
if 'fonte' in df_h_gfc.columns and 'portal_origem' not in df_h_gfc.columns:
    df_h_gfc['portal_origem'] = df_h_gfc['fonte']
df_h_gfc['origem_texto']      = 'AFIRMACAO_CHECADA'
df_h_gfc['fonte_dataset']     = 'GOOGLE_FACTCHECK'
df_h_gfc['referencia_dataset']= df_h_gfc['url_origem'].fillna('')
df_h_gfc['tema']              = ''
df_h_gfc['subtema']           = ''
df_h_gfc['metodo_coleta']     = 'google_factcheck_api'
df_h_gfc['motivo_status']     = ''
df_h_gfc['tamanho_chars']     = df_h_gfc['texto_principal'].fillna('').str.len().astype(str)
df_h_gfc['tamanho_chars_modelo'] = df_h_gfc['texto_principal_modelo'].fillna('').str.len().astype(str)
df_h_gfc['faixa_tamanho_modelo'] = df_h_gfc['tamanho_chars_modelo'].apply(calcular_faixa)
df_h_gfc['label'] = pd.to_numeric(df_h_gfc['label'], errors='coerce').fillna(-1).astype(int)
for col in COLUNAS_FINAIS:
    if col not in df_h_gfc.columns:
        df_h_gfc[col] = ''
df_h_gfc = df_h_gfc[COLUNAS_FINAIS].copy()
df_h_gfc['_fonte_bloco'] = 'GFC'

print(f'V3 harmonizado     : {df_h_v3.shape}  label={df_h_v3["label"].value_counts().to_dict()}')
print(f'Autoral harmonizado: {df_h_aut.shape}  label={df_h_aut["label"].value_counts().to_dict()}')
print(f'GFC harmonizado    : {df_h_gfc.shape}  label={df_h_gfc["label"].value_counts().to_dict()}')

V3 harmonizado     : (10340, 26)  label={1: 5170, 0: 5170}
Autoral harmonizado: (114, 26)  label={1: 114}
GFC harmonizado    : (5145, 26)  label={0: 5110, 1: 35}


## Concatenação e Deduplicação

Dedup em três passos:
1. `texto_principal_modelo` normalizado
2. `url_origem`
3. `texto_principal` normalizado

In [8]:
df_concat = pd.concat([df_h_v3, df_h_aut, df_h_gfc], ignore_index=True)
n_v3  = (df_concat['_fonte_bloco'] == 'V3').sum()
n_aut = (df_concat['_fonte_bloco'] == 'AUTORAL').sum()
n_gfc = (df_concat['_fonte_bloco'] == 'GFC').sum()
n_total = len(df_concat)
print(f'Concat: V3={n_v3}  Autoral={n_aut}  GFC={n_gfc}  TOTAL={n_total}')

df_concat['_chave_modelo']    = df_concat['texto_principal_modelo'].apply(normalizar_texto)
df_concat['_chave_principal'] = df_concat['texto_principal'].apply(normalizar_texto)

antes = len(df_concat)
df_concat = df_concat.drop_duplicates(subset=['_chave_modelo'], keep='first').copy()
n_dedup1  = antes - len(df_concat)

antes = len(df_concat)
mask_url   = df_concat['url_origem'].fillna('').str.strip() != ''
df_concat  = pd.concat([
    df_concat[mask_url].drop_duplicates(subset=['url_origem'], keep='first'),
    df_concat[~mask_url]
], ignore_index=True)
n_dedup2 = antes - len(df_concat)

antes = len(df_concat)
df_concat = df_concat.drop_duplicates(subset=['_chave_principal'], keep='first').copy()
n_dedup3  = antes - len(df_concat)

print(f'Dedup 1 (texto_modelo): -{n_dedup1}')
print(f'Dedup 2 (url_origem) : -{n_dedup2}')
print(f'Dedup 3 (texto norm) : -{n_dedup3}')
print(f'Total dedup          : -{n_dedup1+n_dedup2+n_dedup3}')
print(f'Restantes            : {len(df_concat)}')
print(f'label dist           : {df_concat["label"].value_counts().sort_index().to_dict()}')

Concat: V3=10340  Autoral=114  GFC=5145  TOTAL=15599
Dedup 1 (texto_modelo): -258
Dedup 2 (url_origem) : -64
Dedup 3 (texto norm) : -0
Total dedup          : -322
Restantes            : 15277
label dist           : {0: 9976, 1: 5301}


## Detecção de Conflitos de Label

Textos com mesmo `_chave_modelo` mas labels diferentes são removidos do treino
e salvos em arquivo separado.

In [9]:
df_pre = pd.concat([df_h_v3, df_h_aut, df_h_gfc], ignore_index=True)
df_pre['_chave_modelo'] = df_pre['texto_principal_modelo'].apply(normalizar_texto)

_cnt = (df_pre[df_pre['_chave_modelo'].str.strip() != '']
         .groupby('_chave_modelo')['label'].nunique())
_chaves_conf = set(_cnt[_cnt > 1].index)

df_conflitos    = df_concat[df_concat['_chave_modelo'].isin(_chaves_conf)].copy()
df_sem_conflito = df_concat[~df_concat['_chave_modelo'].isin(_chaves_conf)].copy()
n_conflitos = len(df_conflitos)

print(f'Conflitos de label   : {n_conflitos}')
print(f'Sem conflito         : {len(df_sem_conflito)}')
if n_conflitos > 0:
    print('Exemplos:')
    for ch in list(_chaves_conf)[:3]:
        print(df_pre[df_pre['_chave_modelo']==ch][['texto_principal_modelo','label','dataset_origem']].to_string())

Conflitos de label   : 0
Sem conflito         : 15277


## V4 Full — base completa sem balanceamento

In [10]:
df_v4_full = df_sem_conflito.drop(
    columns=['_chave_modelo','_chave_principal','_fonte_bloco'], errors='ignore'
).copy()
df_v4_full['id_registro'] = [str(uuid.uuid4()) for _ in range(len(df_v4_full))]

print(f'V4 full : {df_v4_full.shape}')
print(f'label   : {df_v4_full["label"].value_counts().sort_index().to_dict()}')
print(f'dataset : {df_v4_full["dataset_origem"].value_counts().head(8).to_dict()}')

V4 full : (15277, 25)
label   : {0: 9976, 1: 5301}
dataset : {'FAKEBR_TEXT_NORMALIZED': 7018, 'CHECKAI_PROPRIO_GFC': 4827, 'FAKETRUEBR': 2828, 'V2_PROPRIA': 490, 'CHECKAI_AUTORAL': 114}


## V4 Balanced — label=0 / label=1 igualados

Amostragem estratificada por `dataset_origem` para preservar diversidade.

In [11]:
_n0  = (df_sem_conflito['label'] == 0).sum()
_n1  = (df_sem_conflito['label'] == 1).sum()
_nmin = min(_n0, _n1)
print(f'label=0={_n0}  label=1={_n1}  min={_nmin}')

def _amostrar_estrat(df_cls, n, seed):
    if len(df_cls) <= n:
        return df_cls
    try:
        return (
            df_cls.groupby('dataset_origem', group_keys=False)
            .apply(lambda g: g.sample(frac=n/len(df_cls), random_state=seed))
            .head(n)
        )
    except Exception:
        return df_cls.sample(n=n, random_state=seed)

_b0 = _amostrar_estrat(df_sem_conflito[df_sem_conflito['label'] == 0], _nmin, SEED)
_b1 = _amostrar_estrat(df_sem_conflito[df_sem_conflito['label'] == 1], _nmin, SEED)

df_v4_balanced = (
    pd.concat([_b0, _b1], ignore_index=True)
    .sample(frac=1, random_state=SEED).reset_index(drop=True)
)
df_v4_balanced = df_v4_balanced.drop(
    columns=['_chave_modelo','_chave_principal','_fonte_bloco'], errors='ignore'
).copy()
df_v4_balanced['id_registro'] = [str(uuid.uuid4()) for _ in range(len(df_v4_balanced))]

print(f'Descartados: label=0={_n0-len(_b0)}  label=1={_n1-len(_b1)}')
print(f'V4 balanced : {df_v4_balanced.shape}')
print(f'label       : {df_v4_balanced["label"].value_counts().sort_index().to_dict()}')
print(f'dataset     : {df_v4_balanced["dataset_origem"].value_counts().head(8).to_dict()}')

label=0=9976  label=1=5301  min=5301
Descartados: label=0=4675  label=1=0
V4 balanced : (10602, 25)
label       : {0: 5301, 1: 5301}
dataset     : {'FAKEBR_TEXT_NORMALIZED': 5408, 'CHECKAI_PROPRIO_GFC': 2575, 'FAKETRUEBR': 2129, 'V2_PROPRIA': 376, 'CHECKAI_AUTORAL': 114}


C:\Users\offan\AppData\Local\Temp\ipykernel_15644\1365809106.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=n/len(df_cls), random_state=seed))


## V4 Text Control — equilibrado por faixa de tamanho

Para cada faixa (`curto`, `medio`, `longo`, `muito_longo`) balanceia label=0 e label=1.

In [12]:
_FAIXAS = ['curto', 'medio', 'longo', 'muito_longo']
_frames_tc = []
_rel_faixas = []

for _faixa in _FAIXAS:
    _sub = df_sem_conflito[df_sem_conflito['faixa_tamanho_modelo'] == _faixa]
    _s0  = _sub[_sub['label'] == 0]
    _s1  = _sub[_sub['label'] == 1]
    _nm  = min(len(_s0), len(_s1))
    if _nm == 0:
        _rel_faixas.append((_faixa, len(_s0), len(_s1), 0, 'sem_dados'))
        continue
    _frames_tc.append(pd.concat([
        _s0.sample(n=_nm, random_state=SEED),
        _s1.sample(n=_nm, random_state=SEED),
    ]))
    _rel_faixas.append((_faixa, len(_s0), len(_s1), _nm*2, 'ok'))

df_v4_tc = (
    pd.concat(_frames_tc, ignore_index=True)
    .sample(frac=1, random_state=SEED).reset_index(drop=True)
)
df_v4_tc = df_v4_tc.drop(
    columns=['_chave_modelo','_chave_principal','_fonte_bloco'], errors='ignore'
).copy()
df_v4_tc['id_registro'] = [str(uuid.uuid4()) for _ in range(len(df_v4_tc))]

print(f'  {"Faixa":<14} {"label=0":>8} {"label=1":>8} {"total":>8}  status')
for _f,_n0,_n1,_tot,_st in _rel_faixas:
    print(f'  {_f:<14} {_n0:>8} {_n1:>8} {_tot:>8}  {_st}')
print(f'V4 text_control : {df_v4_tc.shape}')
print(f'label           : {df_v4_tc["label"].value_counts().sort_index().to_dict()}')

  Faixa           label=0  label=1    total  status
  curto              5893      416      832  ok
  medio              3987     4141     7974  ok
  longo                94      565      188  ok
  muito_longo           2      179        4  ok
V4 text_control : (8998, 25)
label           : {0: 4499, 1: 4499}


## Salvar Arquivos

In [13]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

_ARQ_FULL  = PASTA_SAIDA / f'dataset_final_treino_v4_full_{TIMESTAMP}.csv'
_ARQ_BAL   = PASTA_SAIDA / f'dataset_final_treino_v4_balanced_{TIMESTAMP}.csv'
_ARQ_TC    = PASTA_SAIDA / f'dataset_final_treino_v4_text_control_{TIMESTAMP}.csv'
_ARQ_RELAT = PASTA_SAIDA / f'relatorio_dataset_final_treino_v4_{TIMESTAMP}.md'

df_v4_full.to_csv(_ARQ_FULL,  index=False, encoding='utf-8-sig')
df_v4_balanced.to_csv(_ARQ_BAL, index=False, encoding='utf-8-sig')
df_v4_tc.to_csv(_ARQ_TC,      index=False, encoding='utf-8-sig')

print(f'Salvo: {_ARQ_FULL.name}')
print(f'Salvo: {_ARQ_BAL.name}')
print(f'Salvo: {_ARQ_TC.name}')

_ARQ_CONF = None
if n_conflitos > 0:
    _ARQ_CONF = PASTA_SAIDA / f'dataset_final_treino_v4_conflitos_{TIMESTAMP}.csv'
    df_conflitos.drop(columns=['_chave_modelo','_chave_principal','_fonte_bloco'],
                      errors='ignore').to_csv(_ARQ_CONF, index=False, encoding='utf-8-sig')
    print(f'Salvo: {_ARQ_CONF.name}  ({n_conflitos} conflitos)')
else:
    print('Nenhum conflito — arquivo de conflitos não gerado.')

Salvo: dataset_final_treino_v4_full_2026-05-30_23-29-02.csv
Salvo: dataset_final_treino_v4_balanced_2026-05-30_23-29-02.csv
Salvo: dataset_final_treino_v4_text_control_2026-05-30_23-29-02.csv
Nenhum conflito — arquivo de conflitos não gerado.


## Relatório Final

In [14]:
# ── métricas auxiliares ──────────────────────────────────────────────────
_PROPRIA = {'V2_PROPRIA','V3_PROPRIA','CHECKAI_AUTORAL','CHECKAI_PROPRIO_GFC'}

def _pct_prop(df):
    n = df['dataset_origem'].isin(_PROPRIA).sum()
    return n, 100*n/max(len(df),1)

_v3p, _v3pct = _pct_prop(df_h_v3)
_v4p, _v4pct = _pct_prop(df_v4_full)

def _sz(df, lbl):
    s = pd.to_numeric(df[df['label']==lbl]['tamanho_chars_modelo'], errors='coerce').dropna()
    return (round(s.mean()), int(s.median())) if len(s) else (0, 0)

_m0, _med0 = _sz(df_v4_full, 0)
_m1, _med1 = _sz(df_v4_full, 1)
_acad_pct  = 100*(df_v4_full['pipeline_origem']=='datasets_academicos').sum()/max(len(df_v4_full),1)
_bal_dist  = df_v4_balanced['label'].value_counts().sort_index()
_deseq     = max(df_v4_full['label'].value_counts())/max(min(df_v4_full['label'].value_counts()),1)
_vies_sz   = abs(_m0-_m1) > 500

# ── construir linhas do relatório ────────────────────────────────────────
_L = []
_L.append('# Relatório — Montagem Dataset Final Treino V4')
_L.append('')
_L.append(f'**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
_L.append(f'**Timestamp:** {TIMESTAMP}')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 1. Arquivos Usados')
_L.append('')
_L.append('| Fonte | Arquivo |')
_L.append('|---|---|')
_L.append(f'| V3 balanced | `{ARQ_V3.name}` |')
_L.append(f'| CheckAI Autoral | `{ARQ_AUT.name}` |')
_L.append(f'| GFC Expandido | `{ARQ_GFC.name}` |')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 2. Contagens por Etapa')
_L.append('')
_L.append('| Etapa | Registros |')
_L.append('|---|---|')
_L.append(f'| V3 balanced (base) | {n_v3:,} |')
_L.append(f'| Autoral bruto | {len(df_aut_raw):,} |')
_L.append(f'| Autoral após filtro | {len(df_aut):,} |')
_L.append(f'| GFC bruto | {len(df_gfc_raw):,} |')
_L.append(f'| GFC após filtro | {len(df_gfc):,} |')
_L.append(f'| **Total inicial (concat)** | **{n_total:,}** |')
_L.append(f'| Removidos dedup 1 (texto_modelo) | -{n_dedup1:,} |')
_L.append(f'| Removidos dedup 2 (url_origem) | -{n_dedup2:,} |')
_L.append(f'| Removidos dedup 3 (texto_principal) | -{n_dedup3:,} |')
_L.append(f'| Removidos conflito de label | -{n_conflitos:,} |')
_L.append(f'| **V4 full total** | **{len(df_v4_full):,}** |')
_L.append(f'| **V4 balanced total** | **{len(df_v4_balanced):,}** |')
_L.append(f'| **V4 text_control total** | **{len(df_v4_tc):,}** |')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 3. Distribuição por Label')
_L.append('')
_L.append('### V4 full')
_L.append('```')
_L.append(df_v4_full['label'].value_counts().sort_index().to_string())
_L.append('```')
_L.append('')
_L.append('### V4 balanced')
_L.append('```')
_L.append(_bal_dist.to_string())
_L.append('```')
_L.append('')
_L.append('### V4 text_control')
_L.append('```')
_L.append(df_v4_tc['label'].value_counts().sort_index().to_string())
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 4. Distribuição por dataset_origem (V4 full)')
_L.append('')
_L.append('```')
_L.append(df_v4_full['dataset_origem'].value_counts().to_string())
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 5. Distribuição por pipeline_origem (V4 full)')
_L.append('')
_L.append('```')
_L.append(df_v4_full['pipeline_origem'].value_counts().to_string())
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 6. Distribuição por origem_qualidade (V4 full)')
_L.append('')
_L.append('```')
_L.append(df_v4_full['origem_qualidade'].value_counts().to_string())
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 7. Cobertura Temática (V4 full)')
_L.append('')
_L.append('### tema')
_L.append('```')
_L.append(df_v4_full['tema'].replace('','(vazio)').value_counts().head(20).to_string())
_L.append('```')
_L.append('')
_L.append('### tema_query')
_L.append('```')
_L.append(df_v4_full['tema_query'].replace('','(vazio)').value_counts().head(20).to_string())
_L.append('```')
_L.append('')
_L.append('### subtema_query (top 20)')
_L.append('```')
_L.append(df_v4_full['subtema_query'].replace('','(vazio)').value_counts().head(20).to_string())
_L.append('```')
_L.append('')
_L.append('### Top 20 queries GFC (query_matched)')
_L.append('```')
_vc_q = df_v4_full[df_v4_full['query_matched'].fillna('')!='']['query_matched'].value_counts()
_L.append(_vc_q.head(20).to_string() if len(_vc_q) else 'Nenhuma')
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 8. Participação da Base Própria')
_L.append('')
_L.append('| Versão | Registros próprios | % |')
_L.append('|---|---|---|')
_L.append(f'| V3 balanced | {_v3p:,} | {_v3pct:.1f}% |')
_L.append(f'| V4 full | {_v4p:,} | {_v4pct:.1f}% |')
_L.append(f'| Aumento absoluto | +{_v4p-_v3p:,} | +{_v4pct-_v3pct:.1f}pp |')
_L.append('')
_L.append('*Considera como própria: V2_PROPRIA, V3_PROPRIA, CHECKAI_AUTORAL, CHECKAI_PROPRIO_GFC*')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 9. Análise de Tamanho (V4 full)')
_L.append('')
_L.append('| Métrica | label=0 | label=1 |')
_L.append('|---|---|---|')
_L.append(f'| Média (chars_modelo) | {_m0:,} | {_m1:,} |')
_L.append(f'| Mediana (chars_modelo) | {_med0:,} | {_med1:,} |')
_L.append('')
_L.append('### Faixa por label (V4 full)')
_L.append('```')
_L.append(df_v4_full.groupby(['faixa_tamanho_modelo','label']).size().unstack(fill_value=0).to_string())
_L.append('```')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 10. Alertas Metodológicos')
_L.append('')
_alerta_acad = f'> Acadêmica domina ({_acad_pct:.0f}% datasets_academicos).' if _acad_pct > 80 else f'Base acadêmica = {_acad_pct:.0f}% (aceitável).'
_alerta_prop = 'Base própria aumentou em relação à V3.' if _v4pct > _v3pct else 'Base própria NÃO aumentou em relação à V3.'
_alerta_deseq = f'Desequilíbrio de label ({_deseq:.1f}:1) no V4 full — usar V4 balanced para treino.' if _deseq > 2 else 'Label equilibrado no V4 full.'
_alerta_vies = 'Viés de tamanho detectado entre labels.' if _vies_sz else 'Distribuição de tamanho uniforme entre labels.'
_alerta_conf = f'{n_conflitos} conflitos de label removidos.' if n_conflitos > 0 else 'Nenhum conflito de label.'
for _al in [_alerta_acad, _alerta_prop, _alerta_deseq, _alerta_vies, _alerta_conf]:
    _L.append(f'- {_al}')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 11. Recomendação para Treino')
_L.append('')
_L.append('| Versão | Uso recomendado |')
_L.append('|---|---|')
_L.append('| **V4 balanced** | **Principal — treino e avaliação de F1/accuracy** |')
_L.append('| V4 text_control | Ablação — investigar viés de tamanho de texto |')
_L.append('| V4 full | Exploratório — análise descritiva, não usar para treino direto |')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## Conclusão')
_L.append('')
_L.append('A V4 consolida a base anterior com novas fontes próprias do CheckAI, incluindo '
          'claims verdadeiros autorais rastreáveis e claims falsos/enganosos expandidos via '
          'Google Fact Check. A montagem preserva rastreabilidade, separa qualidade dos '
          'rótulos e mantém versões balanceada e controlada por tamanho para avaliação metodológica.')
_L.append('')
_L.append('---')
_L.append(f'*Gerado por `src/montagem_dataset_final_v4.ipynb` | timestamp `{TIMESTAMP}`*')

relatorio = '\n'.join(_L)
with open(_ARQ_RELAT, 'w', encoding='utf-8') as _f:
    _f.write(relatorio)
print(f'Relatório: {_ARQ_RELAT.name}')

Relatório: relatorio_dataset_final_treino_v4_2026-05-30_23-29-02.md


## Resumo Final

In [15]:
print('=' * 64)
print('  RESUMO — MONTAGEM DATASET FINAL V4')
print('=' * 64)
print()
print('Arquivos gerados:')
print(f'  {_ARQ_FULL.name}')
print(f'  {_ARQ_BAL.name}')
print(f'  {_ARQ_TC.name}')
if _ARQ_CONF:
    print(f'  {_ARQ_CONF.name}')
print(f'  {_ARQ_RELAT.name}')
print()
print(f'Total V4 full         : {len(df_v4_full):>7,}')
print(f'Total V4 balanced     : {len(df_v4_balanced):>7,}')
print(f'Total V4 text_control : {len(df_v4_tc):>7,}')
print()
print('Distribuição V4 balanced:')
for _lbl, _cnt in _bal_dist.items():
    print(f'  label={_lbl} : {_cnt:,}')
print()
_delta_pct = _v4pct - _v3pct
print(f'Base própria V4 full  : {_v4pct:.1f}%  (V3={_v3pct:.1f}%  +{_delta_pct:.1f}pp)')
print()
print('Recomendação: usar V4_balanced para treino principal.')
print('=' * 64)

  RESUMO — MONTAGEM DATASET FINAL V4

Arquivos gerados:
  dataset_final_treino_v4_full_2026-05-30_23-29-02.csv
  dataset_final_treino_v4_balanced_2026-05-30_23-29-02.csv
  dataset_final_treino_v4_text_control_2026-05-30_23-29-02.csv
  relatorio_dataset_final_treino_v4_2026-05-30_23-29-02.md

Total V4 full         :  15,277
Total V4 balanced     :  10,602
Total V4 text_control :   8,998

Distribuição V4 balanced:
  label=0 : 5,301
  label=1 : 5,301

Base própria V4 full  : 35.6%  (V3=4.7%  +30.8pp)

Recomendação: usar V4_balanced para treino principal.
